In [1]:
# Cell 1 — Setup & imports
import os, json, math, pathlib, hashlib, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
sys.path.append("..")

# project modules
from src.transforms import (
    get_preprocessor,
    numeric_features,
    categorical_features,
    add_missing_indicators,
    canonicalize_categorical_df,
    compute_categories_list,    # optional helper; we prefer fitted OHE categories below
)
from src.artifacts import (
    ensure_artifact_dirs,
    save_joblib_with_manifest,
    save_json_with_manifest,
    save_numpy_with_manifest,
    list_manifest,
)

# constants
SEED = 42
DATA_IN = "../data/heart_uci.csv"   # adjust if needed
ART_ROOT = "../artifacts"
PREPROC_META_DIR = os.path.join(ART_ROOT, "preprocessor")
DATA_OUT_DIR = os.path.join(ART_ROOT, "data")

# ensure artifact folders + manifest exist
ensure_artifact_dirs()
os.makedirs(PREPROC_META_DIR, exist_ok=True)
os.makedirs(DATA_OUT_DIR, exist_ok=True)

print("Artifact directories ensured. Manifest at:", os.path.join(ART_ROOT, "manifests", "manifest.json"))


Artifact directories ensured. Manifest at: ../artifacts\manifests\manifest.json


In [2]:
# Cell 2 — Load data and initial EDA
df = pd.read_csv(DATA_IN)
print("Raw shape:", df.shape)
display(df.head())
print("\nInfo:")
display(df.info())
print("\nMissing values per column:")
display(df.isnull().sum())
print("\nTarget distribution (raw 'num'):")
display(df['num'].value_counts(dropna=False))


Raw shape: (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


None


Missing values per column:


id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64


Target distribution (raw 'num'):


num
0    411
1    265
2    109
3    107
4     28
Name: count, dtype: int64

In [3]:
# Cell 3 — Clean target, drop meta columns, save cleaned CSV
# Convert target to binary (0=no disease, 1=disease)
df['num'] = (df['num'] > 0).astype(int)

# 1) create missing indicators for important columns
cols_with_missing = ['ca','thal']   # extend if you find other NaNs
df = add_missing_indicators(df, cols_with_missing)

# 2) canonicalize categorical columns so mixed bool/str types become uniform strings
# categorical_features list is available from src.transforms
df = canonicalize_categorical_df(df, categorical_features)

# Drop obvious metadata columns if present
for c in ['id', 'dataset', 'hospital', 'hospital_name']:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

print("Shape after drop:", df.shape)
print("Data Fields (dtypes):")
print(df.dtypes)

# Save canonical cleaned CSV to artifacts/data for reproducibility (single canonical location)
clean_csv_path = os.path.join(DATA_OUT_DIR, "heart_uci_clean.csv")
os.makedirs(DATA_OUT_DIR, exist_ok=True)
df.to_csv(clean_csv_path, index=False)
print("Saved cleaned CSV:", clean_csv_path)

# Save simple metadata record
meta = {
    "source_csv": os.path.abspath(DATA_IN),
    "clean_csv": os.path.abspath(clean_csv_path),
    "n_rows": int(df.shape[0]),
    "n_cols": int(df.shape[1]),
    "numeric_features_hint": numeric_features,
    "categorical_features_hint": categorical_features,
    "target": "num",
    "created_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}
save_json_with_manifest(meta, os.path.join(DATA_OUT_DIR, "data_metadata.json"), name="data_metadata")


Shape after drop: (920, 16)
Data Fields (dtypes):
age               int64
sex              object
cp               object
trestbps        float64
chol            float64
fbs              object
restecg          object
thalch          float64
exang            object
oldpeak         float64
slope            object
ca              float64
thal             object
num               int64
ca_missing        int64
thal_missing      int64
dtype: object
Saved cleaned CSV: ../artifacts\data\heart_uci_clean.csv


{'name': 'data_metadata',
 'path': '../artifacts\\data\\data_metadata.json',
 'sha256': '7129acff09cc5abbd62dc60a6cb59add7500bdc1e29d3680f4695a1cfe3602ab',
 'created_at': '2026-01-07 03:00:12',
 'notes': ''}

In [4]:
# Cell 4 — Sanity check of categorical canonicalization (use X copy)
try:
    from src.transforms import categorical_features
except Exception:
    categorical_features = ['sex','cp','fbs','restecg','exang','slope','thal']

# Use a copy for checking only
X = df.drop(columns=['num']).copy()

print("=== BEFORE canonicalize (sample from df) ===")
for c in categorical_features:
    if c in X.columns:
        print(c, "dtype:", df[c].dtype, "sample uniques:", pd.Series(df[c].unique()).tolist()[:10])

# X already canonicalized in Cell 3 but re-run on copy to be explicit (idempotent)
X = canonicalize_categorical_df(X, categorical_features)

print("\n=== AFTER canonicalize (on X copy) ===")
for c in categorical_features:
    if c in X.columns:
        print(c, "dtype:", X[c].dtype, "sample uniques:", pd.Series(X[c].unique()).tolist()[:10])

# Reattach target and write the canonical cleaned CSV that downstream notebooks read
df_clean = pd.concat([X, df['num']], axis=1)
out_csv = os.path.join(DATA_OUT_DIR, "heart_uci_clean.csv")
df_clean.to_csv(out_csv, index=False)
print("Saved cleaned CSV with canonicalized categories + missing indicators ->", out_csv)

print("Missing indicators added for:", cols_with_missing)


=== BEFORE canonicalize (sample from df) ===
sex dtype: object sample uniques: ['Male', 'Female']
cp dtype: object sample uniques: ['typical angina', 'asymptomatic', 'non-anginal', 'atypical angina']
fbs dtype: object sample uniques: ['True', 'False', nan]
restecg dtype: object sample uniques: ['lv hypertrophy', 'normal', 'st-t abnormality', nan]
exang dtype: object sample uniques: ['False', 'True', nan]
slope dtype: object sample uniques: ['downsloping', 'flat', 'upsloping', nan]
thal dtype: object sample uniques: ['fixed defect', 'normal', 'reversable defect', nan]

=== AFTER canonicalize (on X copy) ===
sex dtype: object sample uniques: ['Male', 'Female']
cp dtype: object sample uniques: ['typical angina', 'asymptomatic', 'non-anginal', 'atypical angina']
fbs dtype: object sample uniques: ['True', 'False', nan]
restecg dtype: object sample uniques: ['lv hypertrophy', 'normal', 'st-t abnormality', nan]
exang dtype: object sample uniques: ['False', 'True', nan]
slope dtype: object sam

In [5]:
# Cell 5 — Train/Holdout split using canonicalized X
X = df_clean.drop(columns=['num']).copy()
y = df_clean['num'].copy()

X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# canonicalize train split again as an extra safety (idempotent)
X_train = canonicalize_categorical_df(X_train, categorical_features)

print("Train shape:", X_train.shape, "Hold shape:", X_hold.shape)
# Save CSV versions of splits for reproducibility
data_out_dir = os.path.join(ART_ROOT, "data")
os.makedirs(data_out_dir, exist_ok=True)
pd.concat([X_train, y_train], axis=1).to_csv(os.path.join(data_out_dir, "train.csv"), index=False)
pd.concat([X_hold, y_hold], axis=1).to_csv(os.path.join(data_out_dir, "holdout.csv"), index=False)
save_json_with_manifest({"train_rows": int(X_train.shape[0]), "hold_rows": int(X_hold.shape[0])},
                        os.path.join(data_out_dir, "split_info.json"), name="split_info")
print("Saved train & holdout CSVs.")


Train shape: (736, 15) Hold shape: (184, 15)
Saved train & holdout CSVs.


In [6]:
# Cell 6 — Build & fit preprocessor on training set
# (get_preprocessor should return an unfitted ColumnTransformer)
# If your get_preprocessor supports impute_cat_fill_value, pass it; otherwise call without.
try:
    preprocessor = get_preprocessor(impute_cat_fill_value="missing")
except TypeError:
    preprocessor = get_preprocessor()

preprocessor.fit(X_train)          # IMPORTANT: fit only on training data to avoid leakage
print("Preprocessor fitted on training set.")

# Save fitted preprocessor (joblib) using artifacts helper
preproc_path = os.path.join(PREPROC_META_DIR, "preprocessor_initial.joblib")
save_joblib_with_manifest(preprocessor, preproc_path, name="preprocessor_initial",
                          notes=f"Fitted on train rows={len(X_train)}; SEED={SEED}")
print("Saved preprocessor:", preproc_path)


Preprocessor fitted on training set.
Saved preprocessor: ../artifacts\preprocessor\preprocessor_initial.joblib


In [7]:
# Cell 7 — Extract categories_list from fitted preprocessor (correct, leak-free)
cat_pipe = preprocessor.named_transformers_.get("cat")
if cat_pipe is None:
    raise RuntimeError("Preprocessor has no 'cat' transformer — check get_preprocessor() definition")

onehot = cat_pipe.named_steps.get("onehot")
if onehot is None:
    raise RuntimeError("Categorical pipeline has no OneHotEncoder step named 'onehot'")

# Convert categories arrays to list-of-lists of strings
categories_list = [list(map(str, cats)) for cats in onehot.categories_]

json.dump(
    {"categorical_features": categorical_features, "categories_list": categories_list},
    open(os.path.join(PREPROC_META_DIR, "categories_list.json"), "w"),
    indent=2
)

print("Saved categories_list (from fitted preprocessor). Sample:")
for i,c in enumerate(categorical_features):
    print(f" - {c} -> {categories_list[i][:10]}")


Saved categories_list (from fitted preprocessor). Sample:
 - sex -> ['Female', 'Male']
 - cp -> ['asymptomatic', 'atypical angina', 'non-anginal', 'typical angina']
 - fbs -> ['False', 'True', 'missing']
 - restecg -> ['lv hypertrophy', 'normal', 'st-t abnormality']
 - exang -> ['False', 'True', 'missing']
 - slope -> ['downsloping', 'flat', 'missing', 'upsloping']
 - thal -> ['fixed defect', 'missing', 'normal', 'reversable defect']


In [8]:
# Cell 8 — Sanity-transform the holdout and save arrays for quick tests 
X_hold_trans = preprocessor.transform(X_hold)  # dense numpy (depending on transformer config)
print("Hold transformed shape:", X_hold_trans.shape)

# Save transformed arrays (optional but handy)
save_numpy_with_manifest(X_hold_trans, os.path.join(PREPROC_META_DIR, "hold_transformed.npy"),
                         name="hold_transformed", notes="Transformed holdout (dense)")

save_numpy_with_manifest(y_hold.to_numpy(), os.path.join(PREPROC_META_DIR, "hold_y.npy"),
                         name="hold_y", notes="Holdout labels")

# show manifest summary
print("\nCurrent manifest entries:")
list_manifest()


Hold transformed shape: (184, 29)

Current manifest entries:
- test_json : ../artifacts/manifests/test_entry.json (sha256=6ae76eb198b5a8dc19d88d1ff173156852abfc236034f1b6dc5dcb4548c3db8c)
- data_metadata : ../artifacts\data\data_metadata.json (sha256=cfca8d592da1532948ad5d89b6aac85757546f9fc59dbd9934c955dff546481f)
- split_info : ../artifacts\data\split_info.json (sha256=c38aa53501e1a25793c8119687742869d9cfe1ccdb768c250aff36b319fdb195)
- data_metadata : ../artifacts\data\data_metadata.json (sha256=d550a0e495235647beffd1bb4cfdff0a60142863f666a13ea3222e6073c45adc)
- preprocessor_initial : ../artifacts\preprocessor\preprocessor_initial.joblib (sha256=ee2c60e0482a96bd0d766bdf18e61a0b881f8f3cb3db5271121b6e15ccc4c75f)
- preprocessor_initial : ../artifacts\preprocessor\preprocessor_initial.joblib (sha256=ee2c60e0482a96bd0d766bdf18e61a0b881f8f3cb3db5271121b6e15ccc4c75f)
- feature_index_map : ../artifacts\feature_selection\feature_index_map.json (sha256=fa871ad75e4b7b40e2bd31bfc1d5e82788a45f71b

[{'name': 'test_json',
  'path': '../artifacts/manifests/test_entry.json',
  'sha256': '6ae76eb198b5a8dc19d88d1ff173156852abfc236034f1b6dc5dcb4548c3db8c',
  'created_at': '2025-12-07 19:26:43',
  'notes': ''},
 {'name': 'data_metadata',
  'path': '../artifacts\\data\\data_metadata.json',
  'sha256': 'cfca8d592da1532948ad5d89b6aac85757546f9fc59dbd9934c955dff546481f',
  'created_at': '2025-12-07 19:51:05',
  'notes': ''},
 {'name': 'split_info',
  'path': '../artifacts\\data\\split_info.json',
  'sha256': 'c38aa53501e1a25793c8119687742869d9cfe1ccdb768c250aff36b319fdb195',
  'created_at': '2025-12-07 19:51:53',
  'notes': ''},
 {'name': 'data_metadata',
  'path': '../artifacts\\data\\data_metadata.json',
  'sha256': 'd550a0e495235647beffd1bb4cfdff0a60142863f666a13ea3222e6073c45adc',
  'created_at': '2025-12-07 20:01:17',
  'notes': ''},
 {'name': 'preprocessor_initial',
  'path': '../artifacts\\preprocessor\\preprocessor_initial.joblib',
  'sha256': 'ee2c60e0482a96bd0d766bdf18e61a0b881f8f

In [9]:
# Cell 9 — Basic shapes + target balance (must match what you printed)
df_check = pd.read_csv(os.path.join(DATA_OUT_DIR, "heart_uci_clean.csv"))
print("shape:", df_check.shape)
print("target counts:", df_check['num'].value_counts().to_dict())
print("columns:", df_check.columns.tolist())

# missing indicators exist and types
for c in ['ca_missing','thal_missing']:
    print(c, "in df?", c in df_check.columns, "dtype:", df_check[c].dtype if c in df_check.columns else None,
          "unique:", df_check[c].unique()[:10] if c in df_check.columns else None)

# categorical types check
for c in categorical_features:
    if c in df_check.columns:
        print(c, "dtype:", df_check[c].dtype, "sample unique (first 10):", pd.unique(df_check[c].dropna())[:10])


shape: (920, 16)
target counts: {1: 509, 0: 411}
columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'ca_missing', 'thal_missing', 'num']
ca_missing in df? True dtype: int64 unique: [0 1]
thal_missing in df? True dtype: int64 unique: [0 1]
sex dtype: object sample unique (first 10): ['Male' 'Female']
cp dtype: object sample unique (first 10): ['typical angina' 'asymptomatic' 'non-anginal' 'atypical angina']
fbs dtype: object sample unique (first 10): [True False]
restecg dtype: object sample unique (first 10): ['lv hypertrophy' 'normal' 'st-t abnormality']
exang dtype: object sample unique (first 10): [False True]
slope dtype: object sample unique (first 10): ['downsloping' 'flat' 'upsloping']
thal dtype: object sample unique (first 10): ['fixed defect' 'normal' 'reversable defect']


In [10]:
# Cell 10 — print preprocessor summary and get transformed feature names (fit on train)
from src.transforms import get_feature_names_from_preprocessor  # helper in transforms.py

preproc_debug = get_preprocessor()
preproc_debug.fit(pd.read_csv(os.path.join(DATA_OUT_DIR, "train.csv")).drop(columns=["num"]))
names = get_feature_names_from_preprocessor(preproc_debug, pd.read_csv(os.path.join(DATA_OUT_DIR, "train.csv")).drop(columns=["num"]).columns.tolist())
print("Transformed features count:", len(names))
print("First 30 names:", names[:30])
# save names for later debugging
json.dump(names, open(os.path.join(ART_ROOT, "feature_names_debug.json"), "w"))


Transformed features count: 25
First 30 names: ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'sex_Female', 'sex_Male', 'cp_asymptomatic', 'cp_atypical angina', 'cp_non-anginal', 'cp_typical angina', 'fbs_False', 'fbs_True', 'restecg_lv hypertrophy', 'restecg_normal', 'restecg_st-t abnormality', 'exang_False', 'exang_True', 'slope_downsloping', 'slope_flat', 'slope_upsloping', 'thal_fixed defect', 'thal_normal', 'thal_reversable defect']


In [11]:
import joblib, numpy as np, pandas as pd
from src.transforms import canonicalize_categorical_df, categorical_features, get_feature_names_from_preprocessor

# 1) load saved preprocessor
preproc = joblib.load("../artifacts/preprocessor/preprocessor_initial.joblib")
print("Loaded preproc:", preproc)

# 2) load train/hold CSVs AND canonicalize immediately
X_train = pd.read_csv("../artifacts/data/train.csv").drop(columns=["num"])
X_hold  = pd.read_csv("../artifacts/data/holdout.csv").drop(columns=["num"])
X_train = canonicalize_categorical_df(X_train, categorical_features)
X_hold  = canonicalize_categorical_df(X_hold, categorical_features)

# 3) compare shapes and names
names = get_feature_names_from_preprocessor(preproc, X_train.columns.tolist())
print("len(names):", len(names))
print("names[:30]:", names[:30])

Xt = preproc.transform(X_train)
Xh = preproc.transform(X_hold)
print("X_train_trans shape:", Xt.shape)
print("X_hold_trans  shape:", Xh.shape)


Loaded preproc: ColumnTransformer(sparse_threshold=0,
                  transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalch',
                                  'oldpeak', 'ca']),
                                ('cat',
                                 Pipeline(steps=[('cast_to_str',
                                                  FunctionTransformer(func=<function to_string_block at 0x000001AB7719CAE0>)),
                                                 ('imputer',
                                                  SimpleImputer(fill_value='missing',
                                                                strategy='constant')),
                                                 ('onehot',
                             